# 01 — Scrape walkthrough

Run this on a local machine (CPU-only is fine — scraping is I/O-bound). Polite default is 1 req/sec per host; the full provided 3,815 URL run takes ~70 minutes.

Inputs:
- `data/starter/starter_urls.csv` — the course-provided CSV (drop it there before running).

Outputs:
- `data/interim/starter_scraped.csv`
- `data/interim/historical_FoxNews.csv`
- `data/interim/historical_NBC.csv`

In [ ]:
# If you're running on Colab and cloning the repo:
# !git clone https://github.com/<your-team>/cis-4190-project.git
# %cd cis-4190-project

import os
print('CWD:', os.getcwd())

In [ ]:
%pip install -q -r requirements.txt 2>/dev/null || %pip install -q requests beautifulsoup4 lxml tenacity tqdm pandas

## 1. Sanity check: starter CSV is in place

In [ ]:
import pandas as pd
from pathlib import Path

starter = Path('data/starter/starter_urls.csv')
assert starter.exists(), f'Drop the course CSV at {starter} first'
df = pd.read_csv(starter)
print('rows:', len(df))
print('columns:', df.columns.tolist())
df.head()

## 2. Smoke-test the scraper on 5 URLs

In [ ]:
!python -m src.scrape.starter_urls --limit 5 --output data/interim/_smoke.csv
import pandas as pd
pd.read_csv('data/interim/_smoke.csv')

## 3. Full starter-URL scrape (~70 min; resumable)

In [ ]:
!python -m src.scrape.starter_urls --use-wayback

## 4. Historical sitemap expansion (parallel runs OK across team members)

In [ ]:
!python -m src.scrape.sitemap_expand --source FoxNews --years 2019-2024 --per-year 800 --use-wayback

In [ ]:
!python -m src.scrape.sitemap_expand --source NBC --years 2019-2024 --per-year 800 --use-wayback

## 5. Inspect the raw harvest

In [ ]:
import pandas as pd
for name in ['starter_scraped', 'historical_FoxNews', 'historical_NBC']:
    p = f'data/interim/{name}.csv'
    try:
        df = pd.read_csv(p)
        ok = df['fetch_status'].isin(['ok', 'wayback']).sum()
        print(f'{name}: {len(df)} rows, {ok} usable, {len(df)-ok} failed')
    except FileNotFoundError:
        print(f'{name}: not yet generated')